# Product Affinity + RFM Hybrid + Lifecycle mới
**Input:** raw Santander 24 sản phẩm; output notebook 01; frozen Hybrid notebook 03; lifecycle presence mới (New/Active/At-risk/Churned/Win-back). Không dùng cờ hoạt động hoặc nhãn Inactive.

**Scoring:** popularity / coownership / temporal affinity / temporal + lifecycle / temporal + RFM / temporal + cả hai. RFM và lifecycle học tỷ lệ mở từng sản phẩm riêng; chưa học interaction joint bucket. NoEvent không được gán RFM nhưng vẫn dùng temporal/lifecycle.

**Đánh giá:** validation 12/2015, khoá phương pháp trước test 01–04/2016; MAP@7 cùng cohort và label coverage. Chấm 05/2016 → gợi ý 06/2016, không có nhãn 06 trong train. Điểm là ranking score, chưa calibrated probability.

**Chỉnh cell 2, Run All.** Output chính: metadata + group-product rates + bảng benchmark + recommendations có lifecycle/rfm_cluster/status. Các kết quả cũ đã xoá; cần chạy lại với input mới.

**Nâng cấp thử nghiệm:** 7 cấu hình trọng số, lịch sử 3/6 tháng và RFM × lifecycle. Cell 17 chọn bằng validation, cell 19 đánh giá test; tháng cuối dùng đúng window/phương pháp đã khóa. Chạy 3 cửa sổ nên lâu hơn bản trước. Chưa thêm logistic/ranker.

**Phần bổ sung:** Apriori itemset tối đa 3 (luật A&B→C), item2vec SGNS NumPy + loss curve, benchmark cùng cohort và MLflow local. Các hyperparameter cố định trước validation. Bản này chưa FPGrowth/logistic/ranker; không đảm bảo các phương pháp mới thắng temporal.


In [ ]:
from pathlib import Path
from functools import lru_cache
import json,zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=Path('/kaggle/working' if Path('/kaggle/working').exists() else './affinity_work')
RAW_PATH=Path('/kaggle/input/competitions/santander-product-recommendation/train_ver2.csv.zip')
FEATURE_DIR='/kaggle/input/notebooks/thientrantdt/01-r-f-m-analysis' # folder rfm_all_months của notebook 01; None = auto-find
RFM_MODEL_DIR='/kaggle/input/notebooks/thientrantdt/03-rfm-quantile-333-555-1' # output notebook 03 có bucket_specs.json + config.json
LIFECYCLE_DIR='/kaggle/input/notebooks/thientrantdt/05-lifecycle-eda-insights' # output mới: lifecycle_full/customer_lifecycle/lifecycle_YYYY-MM.parquet
RFM_METHOD='Hybrid_REDA_F3_M5'
SHOW_DETAILS=False
TOP_K=7;MIN_PAIR_COUNT=100;MIN_TEMP_EXPOSURE=1000;GROUP_PRIOR=1000.0
VALID_MONTH='2015-12';TEST_MONTHS=['2016-01','2016-02','2016-03','2016-04']
RECOMMEND_MONTH='2016-05';BATCH_SIZE=25000
WRITE_ALL_TEST_RECOMMENDATIONS=False
REBUILD_SNAPSHOT_CACHE=True # False chỉ khi cache từ cùng nguồn raw đã đầy đủ
CACHE=ROOT/'affinity_snapshot_cache';OUT=ROOT/'affinity_outputs_current'
META_CACHE=OUT/'customer_metadata'
for folder in [CACHE,OUT,META_CACHE]:folder.mkdir(parents=True,exist_ok=True)
PRODUCTS = ['ind_ahor_fin_ult1','ind_aval_fin_ult1','ind_cco_fin_ult1','ind_cder_fin_ult1',
'ind_cno_fin_ult1','ind_ctju_fin_ult1','ind_ctma_fin_ult1','ind_ctop_fin_ult1',
'ind_ctpp_fin_ult1','ind_deco_fin_ult1','ind_deme_fin_ult1','ind_dela_fin_ult1',
'ind_ecue_fin_ult1','ind_fond_fin_ult1','ind_hip_fin_ult1','ind_plan_fin_ult1',
'ind_pres_fin_ult1','ind_reca_fin_ult1','ind_tjcr_fin_ult1','ind_valo_fin_ult1',
'ind_viv_fin_ult1','ind_nomina_ult1','ind_nom_pens_ult1','ind_recibo_ult1']
assert len(PRODUCTS)==24

def next_month(m):return str(pd.Period(m,'M')+1)
def prev_month(m):return str(pd.Period(m,'M')-1)
def snapshot(m):return pd.read_parquet(CACHE/f'{m}.parquet')
PRODUCT_MAP = {'ind_ahor_fin_ult1': 'prod_saving_account', 'ind_aval_fin_ult1': 'prod_guarantees', 'ind_cco_fin_ult1': 'prod_current_account', 'ind_cder_fin_ult1': 'prod_derivada_account', 'ind_cno_fin_ult1': 'prod_payroll_account', 'ind_ctju_fin_ult1': 'prod_junior_account', 'ind_ctma_fin_ult1': 'prod_mas_particular_account', 'ind_ctop_fin_ult1': 'prod_particular_account', 'ind_ctpp_fin_ult1': 'prod_particular_plus_account', 'ind_deco_fin_ult1': 'prod_short_term_deposit', 'ind_deme_fin_ult1': 'prod_medium_term_deposit', 'ind_dela_fin_ult1': 'prod_long_term_deposit', 'ind_ecue_fin_ult1': 'prod_eaccount', 'ind_fond_fin_ult1': 'prod_funds', 'ind_hip_fin_ult1': 'prod_mortgage', 'ind_plan_fin_ult1': 'prod_pension_plan', 'ind_pres_fin_ult1': 'prod_loans', 'ind_reca_fin_ult1': 'prod_taxes', 'ind_tjcr_fin_ult1': 'prod_credit_card', 'ind_valo_fin_ult1': 'prod_securities', 'ind_viv_fin_ult1': 'prod_home_account', 'ind_nomina_ult1': 'prod_payroll', 'ind_nom_pens_ult1': 'prod_pension_payroll', 'ind_recibo_ult1': 'prod_direct_debit'}

APRIORI_MIN_SUPPORT=.001
APRIORI_MIN_CONFIDENCE=.05
APRIORI_MAX_LEN=3 # antecedent max 2, one consequent
EMBED_DIM=16;EMBED_EPOCHS=400;EMBED_LR=.03;EMBED_NEGATIVE_K=5;EMBED_SEED=42
ENABLE_MLFLOW=True # if missing, install with %pip install mlflow, then rerun tracking cell


In [ ]:
# Resolve only current pipeline outputs; never auto select old RFM/lifecycle schema.
def resolve_dir(configured,pattern,required_names,label):
    if configured is not None:
        root=Path(configured)
        if all((root/name).exists() for name in required_names):return root
        roots={p.parent for p in root.rglob(pattern)}
    else:
        root=Path('/kaggle/input');roots={p.parent for p in root.rglob(pattern)} if root.exists() else set()
    valid=[d for d in sorted(roots) if all((d/name).exists() for name in required_names)]
    assert len(valid)==1,f'{label}: tìm thấy {len(valid)} thư mục đủ file. Chỉnh đường dẫn cell 2. Candidates: {valid}'
    return valid[0]
required_months=[str(m) for m in pd.period_range('2015-01',RECOMMEND_MONTH,freq='M')]
FEATURE_DIR=resolve_dir(FEATURE_DIR,'rfm_????-??.parquet',[f'rfm_{m}.parquet' for m in required_months],'Features notebook 01')
LIFECYCLE_DIR=resolve_dir(LIFECYCLE_DIR,'lifecycle_????-??.parquet',[f'lifecycle_{m}.parquet' for m in required_months],'Lifecycle mới')
RFM_MODEL_DIR=resolve_dir(RFM_MODEL_DIR,'bucket_specs.json',['bucket_specs.json','config.json'],'Hybrid frozen')
bucket_specs=json.loads((RFM_MODEL_DIR/'bucket_specs.json').read_text())
rfm_config=json.loads((RFM_MODEL_DIR/'config.json').read_text())
assert RFM_METHOD in bucket_specs,f'Không có {RFM_METHOD} trong bucket_specs'
RFM_FIT_MONTH=str(rfm_config['fit_month'])[:7]
FEATURES=['R','F_event_months','M_group_points']
assert rfm_config['features']==FEATURES,'RFM đầu vào không phải R + F_event_months + M_group_points'
EDGES={f:np.asarray(bucket_specs[RFM_METHOD][f],float) for f in FEATURES}
assert all(len(e)>=2 and (np.diff(e)>0).all() for e in EDGES.values())
assert RFM_FIT_MONTH<=VALID_MONTH,'Không dùng model RFM fit tương lai'
print('Features:',FEATURE_DIR,'Hybrid:',RFM_MODEL_DIR,'Lifecycle:',LIFECYCLE_DIR,'Fit:',RFM_FIT_MONTH)


In [ ]:
# Canonical metadata, one row per known customer/month. Read once, cache on disk.
def check_month(frame,m):
    assert frame.customer_id.notna().all() and frame.customer_id.is_unique,'ID trùng/thiếu'
    if 'month_idx' in frame:
        idx=pd.Period(m).year*12+pd.Period(m).month
        assert frame.month_idx.eq(idx).all(),'Sai tháng'
    elif 'month' in frame:assert pd.to_datetime(frame.month.astype(str)).dt.strftime('%Y-%m').eq(m).all()
    else:raise ValueError('Thiếu month_idx/month ở output upstream')

def prepare_metadata(m):
    f=pd.read_parquet(FEATURE_DIR/f'rfm_{m}.parquet',columns=['customer_id','month_idx','R','F_event_months','F_events','M_group_points','products_complete'])
    life=pd.read_parquet(LIFECYCLE_DIR/f'lifecycle_{m}.parquet')
    check_month(f,m);check_month(life,m)
    assert life.lifecycle.isin(['New','Active','At-risk','Churned','Win-back']).all(),'Đang dùng lifecycle cũ hoặc nhãn không hợp lệ'
    assert life.loc[life.lifecycle.isin(['New','Active','Win-back']),'is_observed'].eq(True).all()
    assert life.loc[life.lifecycle.isin(['At-risk','Churned']),'is_observed'].eq(False).all()
    assert set(f.customer_id)==set(life.loc[life.is_observed,'customer_id']),'Lifecycle không phủ đúng mọi snapshot của notebook 01; có thể đang dùng mẫu hoặc phiên khác'
    keep=['customer_id','month_idx','is_observed','missing_months','lifecycle','lifecycle_reason']
    keep += [c for c in ['rule_version','last_available_rfm_cluster','last_available_rfm_age_months'] if c in life]
    d=life[keep].merge(f.drop(columns='month_idx'),on='customer_id',how='left',validate='one_to_one')
    d['as_of_month']=pd.Timestamp(m+'-28');d['is_stale']=~d.is_observed
    d['has_event']=d.F_events.gt(0)&d.is_observed
    d['event_history_group']=np.where(~d.is_observed,'NoCurrentSnapshot',np.where(d.has_event,'HasEvent','NoObservedEvent'))
    d['rfm_cluster']=pd.Series(pd.NA,index=d.index,dtype='string')
    finite=np.isfinite(d[FEATURES].to_numpy(float)).all(axis=1)
    eligible=d.is_observed&d.has_event&d.products_complete.eq(True)&finite
    if eligible.any():
        assert d.loc[eligible,'R'].ge(0).all() and d.loc[eligible,'F_event_months'].ge(1).all()
        assert d.loc[eligible,'M_group_points'].between(0,100).all(),'M_group_points ngoài 0..100'
    if m>=RFM_FIT_MONTH and eligible.any():
        levels=[(pd.cut(d.loc[eligible,f],EDGES[f],labels=False,include_lowest=True).astype(int)+1).astype(str) for f in FEATURES]
        d.loc[eligible,'rfm_cluster']='R'+levels[0]+'_F'+levels[1]+'_M'+levels[2]
        # Validate provided assignments, and compute missing months such as 05/2016 using SAME frozen edges.
        ap=RFM_MODEL_DIR/f'assignments_{RFM_METHOD}_{m}.parquet'
        if ap.exists():
            a=pd.read_parquet(ap)
            if 'method' in a:a=a.loc[a.method.eq(RFM_METHOD)]
            assert a.customer_id.is_unique
            datecol='as_of_month' if 'as_of_month' in a else 'month'
            assert pd.to_datetime(a[datecol].astype(str)).dt.strftime('%Y-%m').eq(m).all()
            check=d[['customer_id','rfm_cluster']].merge(a[['customer_id','cluster']],on='customer_id',validate='one_to_one')
            assert check.rfm_cluster.notna().all() and check.rfm_cluster.eq(check.cluster.astype('string')).all(),'Hybrid assignments không khớp biên frozen'
    d['rfm_status']=np.where(~d.is_observed,'NoCurrentSnapshot',np.where(~d.has_event,'NoObservedEvent',
        np.where(m<RFM_FIT_MONTH,'NotYetFitted',np.where(d.rfm_cluster.notna(),'Assigned','InvalidFeatures'))))
    assert d.loc[d.rfm_status.eq('NoObservedEvent'),'rfm_cluster'].isna().all()
    d.to_parquet(META_CACHE/f'{m}.parquet',index=False)
    return {'month':m,'all_known_n':len(d),'observed_n':int(d.is_observed.sum()),'assigned_n':int(d.rfm_cluster.notna().sum()),
        'no_event_n':int((d.rfm_status=='NoObservedEvent').sum()),'invalid_feature_n':int((d.rfm_status=='InvalidFeatures').sum())}
metadata_coverage=pd.DataFrame([prepare_metadata(m) for m in required_months])
metadata_coverage.to_csv(OUT/'metadata_coverage.csv',index=False);display(metadata_coverage)
@lru_cache(maxsize=2)
def metadata_month(m):return pd.read_parquet(META_CACHE/f'{m}.parquet')


## 0. Danh sách khách và nhãn hiện tại
Lấy lifecycle mới làm bảng nền gồm mọi khách đã biết. RFM chỉ dành cho HasEvent đủ feature; NoObservedEvent có `rfm_cluster` trống, không phải bucket RFM.

Tháng 05/2016 được chấm bằng biên Hybrid đã fit 11/2015; không refit quantile. Giữ metadata lifecycle và RFM ở output gợi ý để phân tích sau. Khách vắng/thiếu danh mục dùng popularity fallback, đánh dấu độ cũ; chưa có benchmark return/churn cho họ.


In [ ]:
prediction_customers=metadata_month(RECOMMEND_MONTH).copy()
prediction_customers['prediction_month']=next_month(RECOMMEND_MONTH)
counts=pd.DataFrame([{'month':RECOMMEND_MONTH,'n_target':len(prediction_customers),
    'n_observed':int(prediction_customers.is_observed.sum()),'n_stale':int(prediction_customers.is_stale.sum()),
    'n_rfm_assigned':int(prediction_customers.rfm_cluster.notna().sum()),
    'n_no_event_observed':int(prediction_customers.rfm_status.eq('NoObservedEvent').sum())}])
display(counts)
for col in ['lifecycle','rfm_status']:display(prediction_customers.groupby(col).size().rename('n').to_frame())
if SHOW_DETAILS:display(prediction_customers.head(20))
prediction_customers.to_parquet(OUT/f'prediction_customers_{RECOMMEND_MONTH}.parquet',index=False)


**Cách đọc:** số khách cần gợi ý được xác định trước khi đọc nhãn tương lai. Phần đánh giá phía sau chỉ giữ khách đủ hai snapshot và báo label coverage. Danh sách preview 20 dòng là mẫu hiển thị; file Parquet chứa toàn bộ khách. Khách stale dùng popularity fallback ở bước gợi ý cuối.


## 1. Raw sản phẩm và output hiện tại
Ba input: notebook 01 (`rfm_all_months`), notebook 03 (`bucket_specs.json`, `config.json`, assignments), lifecycle mới (`lifecycle_YYYY-MM.parquet`). Không dùng `rfm_thresholds.json`, M_n_products quantile hoặc lifecycle có Inactive.

Raw Santander vẫn cần để đọc 24 cờ sản phẩm và tạo nhãn theo từng sản phẩm. NoEvent vẫn gợi ý được bằng danh mục; không ép thành RFM. Trước tháng fit, RFM trống; không dùng biên tương lai để chấm lịch sử. Frozen boundaries chỉ được áp dụng từ fit month trở đi.


In [ ]:
def raw_csv_columns(header):
    """Accept raw Santander or the exact prod_* names from the RFM notebook."""
    names=[str(c).strip().lstrip('\ufeff') for c in header]
    assert len(set(names))==len(names), 'Trùng tên cột sau strip'
    source_by_name=dict(zip(names,header))
    if not {'ncodpers','fecha_dato'}.issubset(names):
        raise ValueError('RAW_PATH cần snapshot có ncodpers, fecha_dato. Header đầu: '+str(names[:12]))
    missing=[];rename={};use=[source_by_name['ncodpers'],source_by_name['fecha_dato']]
    rename.update({source_by_name[k]:k for k in ['ncodpers','fecha_dato']})
    for raw in PRODUCTS:
        source=raw if raw in source_by_name else PRODUCT_MAP[raw] if PRODUCT_MAP[raw] in source_by_name else None
        if source is None:missing.append(raw)
        else:
            use.append(source_by_name[source]);rename[source_by_name[source]]=raw
    if missing:
        raise ValueError(f'RAW_PATH={RAW_PATH}: thiếu {len(missing)}/24 cột sở hữu sản phẩm. '
          'Nếu đang dùng test_ver2.csv, đổi sang train_ver2.csv hoặc train_ver2.csv.zip. '
          'Nếu là file đã xử lý, cần đủ 24 cờ ind_* hoặc prod_* theo PRODUCT_MAP. '
          'Không dùng monthly RFM features thay raw. Header đầu: '+str(names[:12]))
    return use,rename

def build_cache(raw_path, chunksize=300000):
    audit = []
    for old in CACHE.glob('????-??.parquet'):old.unlink()
    for old in CACHE.glob('part_*.parquet'):old.unlink()
    # Chạy lại: xóa riêng cache YYYY-MM do notebook này tạo, tránh append trùng.
    parts = {}
    handle = zipfile.ZipFile(raw_path) if str(raw_path).endswith('.zip') else None
    if handle:
        members = [n for n in handle.namelist() if n.endswith('.csv')]
        assert len(members) == 1, members
        stream = handle.open(members[0])
    else: stream = raw_path
    try:
        header=pd.read_csv(stream,nrows=0).columns.tolist()
        usecols,rename=raw_csv_columns(header)
        if handle:
            stream.close();stream=handle.open(members[0])
        else:stream=raw_path
        print('Raw schema hợp lệ: 24 cờ sản phẩm; file:',raw_path)
        for i, chunk in enumerate(pd.read_csv(stream, usecols=usecols,
                                              chunksize=chunksize, low_memory=False)):
            chunk=chunk.rename(columns=rename)
            chunk['month'] = pd.to_datetime(chunk.fecha_dato, errors='raise').dt.strftime('%Y-%m')
            for m, d in chunk.groupby('month', sort=False):
                x = d[PRODUCTS].apply(pd.to_numeric, errors='coerce')
                valid = x.isin([0,1]).all(axis=1)
                audit.append({'month':m,'n_raw':len(d),'n_complete':int(valid.sum())})
                good = x.loc[valid].astype('uint8')
                good.insert(0,'customer_id', pd.to_numeric(d.loc[valid,'ncodpers'],errors='raise').astype('int64'))
                p = CACHE / f'part_{m}_{i}.parquet'; good.to_parquet(p,index=False)
                parts.setdefault(m,[]).append(p)
        for m, paths in sorted(parts.items()):
            d = pd.concat([pd.read_parquet(p) for p in paths],ignore_index=True)
            assert not d.customer_id.duplicated().any(), f'Duplicate customer–month: {m}'
            d.sort_values('customer_id').to_parquet(CACHE / f'{m}.parquet',index=False)
            for p in paths: p.unlink()
    finally:
        if handle: handle.close()
    a = pd.DataFrame(audit).groupby('month',as_index=False)[['n_raw','n_complete']].sum()
    a['complete_coverage'] = a.n_complete / a.n_raw
    a.to_csv(OUT/'snapshot_coverage.csv',index=False)
    return a

raw_identity={'path':str(RAW_PATH),'size':RAW_PATH.stat().st_size,'mtime':RAW_PATH.stat().st_mtime}
if REBUILD_SNAPSHOT_CACHE:
    coverage=build_cache(RAW_PATH)
    (CACHE/'source_identity.json').write_text(json.dumps(raw_identity))
else:
    assert (CACHE/'source_identity.json').exists() and json.loads((CACHE/'source_identity.json').read_text())==raw_identity,'Cache không có fingerprint cùng nguồn raw'
    assert all((CACHE/f'{m}.parquet').exists() for m in required_months),'Cache thiếu tháng'
    coverage=pd.read_csv(OUT/'snapshot_coverage.csv')
assert all((CACHE/f'{m}.parquet').exists() for m in required_months)
display(coverage)


**Cách đọc:** complete_coverage là tỷ lệ snapshot đủ 24 cờ để phân tích. Khách bị lọc chưa được coi là không sở hữu sản phẩm. Cache chỉ cần dựng lại khi thay raw; có thể thay ô trên bằng đọc snapshot_coverage.csv khi chạy lại.

In [ ]:
def join_optional(m,base):
    d=base.merge(metadata_month(m),on='customer_id',how='left',validate='one_to_one')
    assert d.lifecycle.notna().all(),'Metadata thiếu khách'
    assert d.is_observed.eq(True).all(),'Raw complete snapshot không khớp observed metadata'
    assert d.products_complete.eq(True).all(),'Raw và feature khác độ đầy đủ sản phẩm'
    return d

def pair(m):
    a,b=snapshot(m),snapshot(next_month(m))
    d=a.merge(b,on='customer_id',suffixes=('_t','_next'),validate='one_to_one')
    X=d[[p+'_t' for p in PRODUCTS]].to_numpy(dtype=np.float64)
    Z=d[[p+'_next' for p in PRODUCTS]].to_numpy(dtype=np.float64)
    Y=((X==0)&(Z==1)).astype(np.float64)
    return d.customer_id.to_numpy(),X,Y,len(a)


## 2. Ma trận đồng sở hữu và temporal affinity
Support/confidence/lift dùng snapshot tháng chốt. Temporal affinity dùng các cặp lịch sử mà tháng nhãn ≤ tháng chốt: P(mở B tháng sau | có A, chưa có B). Khách xuất hiện nhiều kỳ được đếm theo khách–tháng, không phải khách duy nhất. Temporal confidence là tỷ lệ nhóm, chưa phải xác suất cá nhân.

Popularity dùng acquisition rate trên khách chưa sở hữu từng sản phẩm (khác tỷ lệ sở hữu). Các nhóm lifecycle/RFM dùng tỷ lệ mở sản phẩm có smoothing về tỷ lệ chung; không fit một ma trận cho từng tổ hợp nhóm.

In [ ]:
def coownership(X, min_count=MIN_PAIR_COUNT):
    n=len(X); counts=X.T@X; owners=X.sum(axis=0)
    support=counts/max(n,1)
    confidence=np.divide(counts, owners[:,None],out=np.zeros_like(counts),where=owners[:,None]>0)
    prevalence=owners/max(n,1)
    lift=np.divide(confidence,prevalence[None,:],out=np.zeros_like(counts),where=prevalence[None,:]>0)
    score=np.where(counts>=min_count,confidence*np.log1p(lift),0)
    np.fill_diagonal(score,0)
    return dict(count=counts,support=support,confidence=confidence,lift=lift,score=score)

def fit_at(cutoff,window=None):
    months=sorted(p.stem for p in CACHE.glob('????-??.parquet') if p.stem<cutoff)
    if window is not None:
        lower=str(pd.Period(cutoff,'M')-window)
        months=[m for m in months if m>=lower]
    success=np.zeros(24); exposure=np.zeros(24)
    ts=np.zeros((24,24)); te=np.zeros((24,24)); groups={}
    audits=[]
    for m in months:
        # No label after cutoff: pair m -> m+1 only
        if next_month(m)>cutoff: continue
        ids,X,Y,n_origin=pair(m)
        success+=Y.sum(0); exposure+=(1-X).sum(0)
        ts+=X.T@Y; te+=X.T@(1-X)
        meta=join_optional(m,pd.DataFrame({'customer_id':ids}))
        meta['rfm_lifecycle']=meta.rfm_cluster.astype('string')+' | '+meta.lifecycle.astype('string')
        for col in ['lifecycle','rfm_cluster','rfm_lifecycle']:
            for label, ix in meta.groupby(col).indices.items():
                if pd.isna(label):continue
                key=(col,str(label))
                if key not in groups: groups[key]=[np.zeros(24),np.zeros(24)]
                groups[key][0]+=Y[ix].sum(0);groups[key][1]+=(1-X[ix]).sum(0)
        audits.append({'origin_month':m,'label_month':next_month(m),'n_origin_complete':n_origin,
                       'n_paired':len(ids),'label_coverage':len(ids)/max(n_origin,1)})
    assert audits, 'Không có cặp train'
    pop=np.divide(success,exposure,out=np.zeros(24),where=exposure>0)
    temporal=np.divide(ts+GROUP_PRIOR*pop[None,:],te+GROUP_PRIOR)
    temporal[te<MIN_TEMP_EXPOSURE]=0
    np.fill_diagonal(temporal,0)
    rates={k:(s+GROUP_PRIOR*pop)/(e+GROUP_PRIOR) for k,(s,e) in groups.items()}
    co=coownership(snapshot(cutoff)[PRODUCTS].to_numpy(dtype=np.float64))
    return dict(cutoff=cutoff,window=window,pop=pop,co=co,temporal=temporal,temporal_success=ts,
                temporal_exposure=te,groups=rates,audit=pd.DataFrame(audits))

def export_model(model):
    folder=OUT/model['cutoff'];folder.mkdir(exist_ok=True)
    for k,v in model['co'].items():
        pd.DataFrame(v,index=PRODUCTS,columns=PRODUCTS).to_csv(folder/f'coownership_{k}.csv')
    for k in ['temporal','temporal_success','temporal_exposure']:
        pd.DataFrame(model[k],index=PRODUCTS,columns=PRODUCTS).to_csv(folder/f'{k}.csv')
    model['audit'].to_csv(folder/'training_coverage.csv',index=False)
    pd.DataFrame({'product':PRODUCTS,'acquisition_rate':model['pop']}).to_csv(folder/'popularity.csv',index=False)
    rows=[]
    for (col,label),rate in model['groups'].items():
        for product,value in zip(PRODUCTS,rate):rows.append({'group_type':col,'group':label,'product':product,'rate':value})
    pd.DataFrame(rows).to_csv(folder/'group_product_rates.csv',index=False)
    return folder


from itertools import combinations

def apriori_rules(X):
    X=X.astype(bool);N=len(X);minimum=max(MIN_PAIR_COUNT,int(np.ceil(APRIORI_MIN_SUPPORT*N)))
    freq={};level=[]
    for i in range(24):
        count=int(X[:,i].sum())
        if count>=minimum:freq[(i,)]=count;level.append((i,))
    for size in range(2,APRIORI_MAX_LEN+1):
        candidates=set()
        for a,b in combinations(level,2):
            c=tuple(sorted(set(a)|set(b)))
            if len(c)==size and all(tuple(sub) in freq for sub in combinations(c,size-1)):candidates.add(c)
        new=[]
        for c in sorted(candidates):
            count=int(X[:,c].all(axis=1).sum())
            if count>=minimum:freq[c]=count;new.append(c)
        level=new
        if not level:break
    rows=[]
    for items,count in freq.items():
        if len(items)<2:continue
        for consequent in items:
            antecedent=tuple(i for i in items if i!=consequent)
            confidence=count/freq[antecedent];lift=confidence/(freq[(consequent,)]/N)
            if confidence>=APRIORI_MIN_CONFIDENCE:
                rows.append({'antecedent':antecedent,'consequent':consequent,'n':count,
                    'support':count/N,'confidence':confidence,'lift':lift})
    return pd.DataFrame(rows,columns=['antecedent','consequent','n','support','confidence','lift'])

def train_item2vec(counts):
    # SGNS item-item objective: exact expectation over noise distribution, not a sampled corpus/Gensim replica.
    positive=counts.copy().astype(float);np.fill_diagonal(positive,0)
    support=positive.sum(axis=1);active=support>0
    if positive.sum()==0:return np.zeros((24,24)),np.zeros((24,EMBED_DIM)),[]
    P=positive/positive.sum();noise=support**.75;noise/=noise.sum()
    negative=EMBED_NEGATIVE_K*P.sum(axis=1)[:,None]*noise[None,:]
    rng=np.random.default_rng(EMBED_SEED)
    U=rng.normal(0,.1,(24,EMBED_DIM));V=rng.normal(0,.1,(24,EMBED_DIM))
    mu=np.zeros_like(U);vu=np.zeros_like(U);mv=np.zeros_like(V);vv=np.zeros_like(V);loss=[]
    for epoch in range(1,EMBED_EPOCHS+1):
        logits=U@V.T;sigmoid=1/(1+np.exp(-np.clip(logits,-40,40)))
        loss.append(float((P*np.logaddexp(0,-logits)+negative*np.logaddexp(0,logits)).sum()))
        D=P*(sigmoid-1)+negative*sigmoid
        gu=D@V;gv=D.T@U
        mu=.9*mu+.1*gu;vu=.999*vu+.001*gu*gu
        mv=.9*mv+.1*gv;vv=.999*vv+.001*gv*gv
        U-=EMBED_LR*(mu/(1-.9**epoch))/(np.sqrt(vu/(1-.999**epoch))+1e-8)
        V-=EMBED_LR*(mv/(1-.9**epoch))/(np.sqrt(vv/(1-.999**epoch))+1e-8)
    U[~active]=0
    normalized=U/np.maximum(np.linalg.norm(U,axis=1,keepdims=True),1e-12)
    similarity=np.maximum(normalized@normalized.T,0);similarity[~active,:]=0;similarity[:,~active]=0
    np.fill_diagonal(similarity,0)
    return similarity,U,loss

core_fit_at=fit_at
core_export_model=export_model
def fit_at(cutoff,window=None):
    model=core_fit_at(cutoff,window)
    # One current snapshot/customer, same as association baseline; no next-month label used.
    if window is None:
        X=snapshot(cutoff)[PRODUCTS].to_numpy(float)
        model['apriori']=apriori_rules(X)
        sim,embedding,loss=train_item2vec(model['co']['count'])
        model.update(item2vec_similarity=sim,item2vec_embedding=embedding,item2vec_loss=loss)
    return model

def export_model(model):
    folder=core_export_model(model)
    if 'apriori' in model:
        rules=model['apriori'].copy()
        rules['antecedent']=rules.antecedent.map(lambda x:' & '.join(PRODUCTS[i] for i in x))
        rules['consequent']=rules.consequent.map(lambda x:PRODUCTS[x])
        rules.to_csv(folder/'apriori_rules.csv',index=False)
        pd.DataFrame(model['item2vec_embedding'],index=PRODUCTS).to_csv(folder/'item2vec_embeddings.csv')
        pd.DataFrame(model['item2vec_similarity'],index=PRODUCTS,columns=PRODUCTS).to_csv(folder/'item2vec_similarity.csv')
        pd.DataFrame({'epoch':range(1,len(model['item2vec_loss'])+1),'loss':model['item2vec_loss']}).to_csv(folder/'item2vec_loss.csv',index=False)
    return folder
validation_model=fit_at(VALID_MONTH)
export_model(validation_model)
mat=validation_model['co']['lift'].copy();np.fill_diagonal(mat,np.nan)
fig,ax=plt.subplots(figsize=(9,8));im=ax.imshow(np.clip(mat,0,10),cmap='viridis')
ax.set_xticks(range(24),PRODUCTS,rotation=90,fontsize=7);ax.set_yticks(range(24),PRODUCTS,fontsize=7)
ax.set_title('Đồng sở hữu: lift (màu giới hạn 10; CSV giữ giá trị gốc)');fig.colorbar(im,ax=ax);plt.tight_layout();plt.show()
idx=np.triu_indices(24,1)
rules=pd.DataFrame({'A':np.array(PRODUCTS)[idx[0]],'B':np.array(PRODUCTS)[idx[1]],
                    'n_pair':validation_model['co']['count'][idx],'lift':mat[idx]})
display(rules.query('n_pair >= @MIN_PAIR_COUNT').sort_values('lift',ascending=False).head(12))

print('Apriori rules:',len(validation_model['apriori']))
display(validation_model['apriori'].sort_values('lift',ascending=False).head(10))
fig,ax=plt.subplots(figsize=(7,3));ax.plot(validation_model['item2vec_loss']);ax.set(title='Item2vec SGNS training objective',xlabel='Epoch',ylabel='Loss');plt.tight_layout();plt.show()


**Đọc affinity:** coownership cho biết đồng sở hữu; temporal đo mở sản phẩm tháng kế tiếp từ danh mục tại t. Lift đồng sở hữu cao chưa đủ để khẳng định gợi ý tốt. Tỷ lệ theo nhóm RFM/lifecycle được xuất trong `group_product_rates.csv` để giải thích sản phẩm nào có điểm cao ở nhóm nào.


In [ ]:
METHODS=['popularity','association','temporal','temporal_lifecycle','temporal_rfm','temporal_both']
def score_batch(model,X,meta,method):
    pop=np.broadcast_to(model['pop'],X.shape).copy()
    if method=='popularity': score=pop
    elif method=='association':
        raw=X@model['co']['score']
        score=raw+0.05*pop  # fixed ranking heuristic, not probability
    else:
        assert method in METHODS
        raw=X@model['temporal']/np.maximum(X.sum(1,keepdims=True),1)
        score=0.5*raw+0.5*pop
        cols={'temporal_lifecycle':['lifecycle'],'temporal_rfm':['rfm_cluster'],
              'temporal_both':['lifecycle','rfm_cluster']}.get(method,[])
        total=np.zeros_like(pop);component_n=np.zeros(len(X),dtype=int)
        for col in cols:
            for label,ix in meta.groupby(col).indices.items():
                rate=model['groups'].get((col,str(label)))
                if rate is not None:
                    total[ix]+=rate;component_n[ix]+=1
        has=component_n>0
        # No RFM group (e.g. NoEvent) contributes no fabricated RFM score.
        score[has]=0.5*score[has]+0.5*total[has]/component_n[has,None]
    score[X==1]=-np.inf
    return score

def rank_scores(score,k=TOP_K):
    order=np.argsort(-score,axis=1,kind='stable')[:,:k]
    values=np.take_along_axis(score,order,axis=1)
    return np.where(np.isfinite(values),order,-1),values

def per_customer_metrics(Y,ranks):
    safe=np.maximum(ranks,0)
    hits=np.take_along_axis(Y,safe,axis=1)*(ranks>=0)
    n_true=Y.sum(1)
    precision=np.cumsum(hits,axis=1)/np.arange(1,ranks.shape[1]+1)
    ap=(precision*hits).sum(1)/np.maximum(np.minimum(n_true,ranks.shape[1]),1)
    recall=hits.sum(1)/np.maximum(n_true,1)
    return ap,recall,n_true

def evaluate(m,model,methods):
    assert model['cutoff']==m
    ids,X,Y,n_origin=pair(m)
    meta=join_optional(m,pd.DataFrame({'customer_id':ids}))
    summaries=[];by_group=[]
    for method in methods:
        ap_parts=[];rec_parts=[];true_parts=[];length_parts=[];items=set()
        for start in range(0,len(ids),BATCH_SIZE):
            sl=slice(start,start+BATCH_SIZE)
            ranks,values=rank_scores(score_batch(model,X[sl],meta.iloc[sl].reset_index(drop=True),method))
            ap,rec,nt=per_customer_metrics(Y[sl],ranks)
            ap_parts.append(ap);rec_parts.append(rec);true_parts.append(nt);length_parts.append((ranks>=0).sum(1))
            items.update(ranks[ranks>=0].tolist())
            if WRITE_ALL_TEST_RECOMMENDATIONS:write_recs(m,method,ids[sl],ranks,values,start)
        ap=np.concatenate(ap_parts);rec=np.concatenate(rec_parts);nt=np.concatenate(true_parts);length=np.concatenate(length_parts)
        pos=nt>0
        summaries.append(dict(month=m,method=method,n_evaluated=len(ids),label_coverage=len(ids)/max(n_origin,1),
                              acquisition_prevalence=float(pos.mean()),MAP7_all=float(ap.mean()),
                              MAP7_acquirers=float(ap[pos].mean()) if pos.any() else np.nan,
                              recall7_acquirers=float(rec[pos].mean()) if pos.any() else np.nan,
                              full7_coverage=float((length==TOP_K).mean()),item_coverage=len(items)/24))
        for col in ['lifecycle','rfm_cluster','event_history_group']:
            for label,ix in meta.groupby(col).indices.items():
                by_group.append(dict(month=m,method=method,group_type=col,group=str(label),n=len(ix),
                                     MAP7_all=float(ap[ix].mean()),acquisition_prevalence=float(pos[ix].mean())))
    return pd.DataFrame(summaries),pd.DataFrame(by_group)

def write_recs(m,method,ids,ranks,values,part):
    lists=[[PRODUCTS[j] for j in row if j>=0] for row in ranks]
    scores=[[float(v) for j,v in zip(row,vals) if j>=0] for row,vals in zip(ranks,values)]
    folder=OUT/'product_recommendation'/m/method;folder.mkdir(parents=True,exist_ok=True)
    rec=pd.DataFrame({'customer_id':ids,'as_of_month':m,'method':method,'products':lists,'ranking_scores':scores})
    meta=metadata_month(m)[['customer_id','lifecycle','rfm_cluster','rfm_status','event_history_group']]
    rec=rec.merge(meta,on='customer_id',how='left',validate='one_to_one')
    rec.to_parquet(folder/f'part_{part:07d}.parquet',index=False)

# Prespecified candidates; validation chooses, test only evaluates.
UPGRADE_SPECS={
 'blend80':{'window':None,'weights':(.8,.1,.1,0)},
 'blend60':{'window':None,'weights':(.6,.2,.2,0)},
 'joint20':{'window':None,'weights':(.6,.1,.1,.2)},
 'recent3_temporal':{'window':3,'weights':(1,0,0,0)},
 'recent6_temporal':{'window':6,'weights':(1,0,0,0)},
 'recent3_joint20':{'window':3,'weights':(.6,.1,.1,.2)},
 'recent6_joint20':{'window':6,'weights':(.6,.1,.1,.2)}}
original_score_batch=score_batch
METHODS+=list(UPGRADE_SPECS)
def score_batch(model,X,meta,method):
    if method not in UPGRADE_SPECS:return original_score_batch(model,X,meta,method)
    weights=UPGRADE_SPECS[method]['weights']
    base=original_score_batch(model,X,meta,'temporal')
    # Unmask while blending; owned products excluded again at the end.
    base=np.where(X==1,0,base)
    total=weights[0]*base;denom=np.full(len(X),weights[0],float)
    labels={'rfm_cluster':meta.rfm_cluster.astype('string'),
            'lifecycle':meta.lifecycle.astype('string'),
            'rfm_lifecycle':meta.rfm_cluster.astype('string')+' | '+meta.lifecycle.astype('string')}
    for col,weight in zip(['rfm_cluster','lifecycle','rfm_lifecycle'],weights[1:]):
        if weight==0:continue
        for label in labels[col].dropna().unique():
            ix=np.flatnonzero(labels[col].eq(label).fillna(False).to_numpy(bool))
            rate=model['groups'].get((col,str(label)))
            if rate is not None:total[ix]+=weight*rate;denom[ix]+=weight
    score=total/denom[:,None];score[X==1]=-np.inf
    return score

previous_score_batch=score_batch
METHODS+=['apriori','item2vec']
def score_batch(model,X,meta,method):
    if method not in ['apriori','item2vec']:return previous_score_batch(model,X,meta,method)
    pop=np.broadcast_to(model['pop'],X.shape)
    if method=='item2vec':
        # similarity and popularity have different scales; fixed heuristic, evaluate against baseline.
        raw=X@model['item2vec_similarity']/np.maximum(X.sum(axis=1,keepdims=True),1)
        score=.8*raw+.2*pop
    else:
        score=.05*pop.copy()
        for row in model['apriori'].itertuples():
            match=(X[:,row.antecedent]==1).all(axis=1)
            score[match,row.consequent]+=row.confidence*np.log1p(row.lift)
    score[X==1]=-np.inf
    return score


## 3. Validation và khóa lựa chọn trước test
MAP7_all gồm khách không acquisition (AP=0); MAP7_acquirers là chỉ số bổ sung, không thay thế mẫu số chính. Full7 coverage là tỷ lệ khách có đủ 7 ứng viên; item coverage là số sản phẩm xuất hiện trong gợi ý / 24. Label coverage là tỷ lệ khách snapshot đầy đủ tại t có snapshot đầy đủ tại t+1; không đại diện cho mọi khách mất quan sát.

Cách scoring và smoothing đã cố định trong cấu hình. Nếu chỉnh chúng sau validation, phải khóa lại trước test. Không chọn mô hình theo test.
Window 3/6 tính theo tháng nguồn của cặp lịch sử: chỉ dùng label đã biết tại cutoff. Joint group thiếu/nhỏ được smoothing về popularity; thiếu nhóm thì bỏ thành phần và chuẩn hóa lại trọng số. NoEvent không được tạo joint RFM giả. Các cấu hình là giả thuyết cần kiểm chứng, không đảm bảo tăng MAP.


In [ ]:
available=['popularity','association','apriori','item2vec','temporal','temporal_lifecycle','temporal_rfm','temporal_both']+list(UPGRADE_SPECS)
val_parts=[];val_group_parts=[]
for window in [None,3,6]:
    model=validation_model if window is None else fit_at(VALID_MONTH,window)
    methods=[m for m in available if UPGRADE_SPECS.get(m,{}).get('window')==window]
    a,b=evaluate(VALID_MONTH,model,methods);val_parts.append(a);val_group_parts.append(b)
val=pd.concat(val_parts,ignore_index=True);val_groups=pd.concat(val_group_parts,ignore_index=True)
val.to_csv(OUT/'validation_metrics.csv',index=False);val_groups.to_csv(OUT/'validation_groups.csv',index=False)
display(val.sort_values('MAP7_all',ascending=False))
best=val.sort_values('MAP7_all',ascending=False).iloc[0]
base=val.loc[val.method.eq('temporal'),'MAP7_all'].iloc[0]
SELECTED_METHOD=best['method'] if best.MAP7_all>base+.0001 else 'temporal'
SELECTED_WINDOW=UPGRADE_SPECS.get(SELECTED_METHOD,{}).get('window')
(OUT/'selection.json').write_text(json.dumps({'validation_month':VALID_MONTH,'selected_method':SELECTED_METHOD,
 'window':SELECTED_WINDOW,'candidates':UPGRADE_SPECS,'min_gain_vs_temporal':.0001},indent=2))
print('Khóa trước test:',SELECTED_METHOD,'window=',SELECTED_WINDOW)

**Đọc validation:** so phần bổ sung RFM/lifecycle với temporal. Nhãn Hybrid bắt đầu từ tháng fit 11/2015, nên ít tháng nguồn có lịch sử nhóm lúc validation. Smoothing giúp nhóm nhỏ backoff. Khoá lựa chọn trên validation; không đổi phương pháp theo test.


In [ ]:
test=[];test_groups=[]
for m in TEST_MONTHS:
    for window in [None,3,6]:
        model=fit_at(m,window)
        methods=[method for method in available if UPGRADE_SPECS.get(method,{}).get('window')==window]
        a,b=evaluate(m,model,methods);test.append(a);test_groups.append(b)
    print('Đã đánh giá',m)
metrics=pd.concat(test,ignore_index=True);groups=pd.concat(test_groups,ignore_index=True)
assert metrics.groupby('month').n_evaluated.nunique().eq(1).all()
metrics.to_csv(OUT/'test_metrics.csv',index=False);groups.to_csv(OUT/'test_groups.csv',index=False)
summary=metrics.groupby('method')[['MAP7_all','MAP7_acquirers','recall7_acquirers','item_coverage']].mean()
display(summary.sort_values('MAP7_all',ascending=False).round(6))
reference=metrics.loc[metrics.method.eq('temporal')].set_index('month').MAP7_all
metrics['delta_vs_temporal']=metrics.MAP7_all-metrics.month.map(reference)
metrics.to_csv(OUT/'test_metrics.csv',index=False)
print('Phương pháp đã khóa:',SELECTED_METHOD,'; không đổi lựa chọn theo bảng test.')
fig,ax=plt.subplots(figsize=(10,4))
for method in dict.fromkeys(['popularity','temporal',SELECTED_METHOD]):
    d=metrics.loc[metrics.method.eq(method)];ax.plot(d.month,d.MAP7_all,marker='o',label=method)
ax.set_ylabel('MAP@7 mọi khách đủ nhãn');ax.legend();plt.tight_layout();plt.show()


**Nhận xét sau test:** phương pháp đã khóa có thắng baseline qua các tháng không? Lợi ích có tập trung ở một nhóm lớn? Luôn đọc n và acquisition_prevalence trước khi so MAP giữa các nhóm. Báo cáo test trung thực kể cả giảm điểm; không đổi selection theo test. Mean theo tháng trong bảng khác pooled mean theo khách nếu các tháng có kích thước khác nhau.

## 4. Gợi ý tháng cuối cho mọi khách đã biết
Metadata nền là lifecycle mới. Có danh mục raw đầy đủ hiện tại → model đã chọn, loại sản phẩm đang sở hữu. Danh mục thiếu/cũ → popularity fallback và đánh dấu catalog_status.

Khách NoObservedEvent: RFM trống, vẫn gợi ý từ danh mục và lifecycle. Khách vắng: RFM hiện tại trống; không dùng bucket cũ như hiện tại. Frozen Hybrid tự chấm tháng 05/2016; không phải refit. Đánh giá hiện tại chỉ trên hai snapshot đầy đủ liên tiếp, không đại diện cho hiệu quả return/churn.


In [ ]:
final_model=fit_at(RECOMMEND_MONTH,SELECTED_WINDOW);export_model(final_model)
current=snapshot(RECOMMEND_MONTH)
assert RECOMMEND_MONTH == prediction_customers.as_of_month.dt.strftime('%Y-%m').iloc[0]
universe=prediction_customers[['customer_id']].copy()
assert not universe.customer_id.duplicated().any()
last=pd.DataFrame(columns=PRODUCTS+['catalog_month'])
last.index.name='customer_id'
for p in sorted(CACHE.glob('????-??.parquet')):
    if p.stem>RECOMMEND_MONTH:continue
    frame=pd.read_parquet(p).set_index('customer_id')
    frame['catalog_month']=p.stem
    last=pd.concat([last.loc[~last.index.isin(frame.index)],frame])
for start in range(0,len(universe),BATCH_SIZE):
    ids=universe.customer_id.iloc[start:start+BATCH_SIZE].to_numpy()
    catalog=last.reindex(ids)
    X=catalog[PRODUCTS].apply(pd.to_numeric,errors='coerce').fillna(0).to_numpy(dtype=np.float64)
    status=np.where(catalog.catalog_month.eq(RECOMMEND_MONTH),'current_complete',
                    np.where(catalog.catalog_month.notna(),'last_complete_snapshot','no_complete_snapshot')).tolist()
    # Chỉ check raw–RFM is_observed trên current; metadata all-customer không được assert observed.
    current_ids=set(current.customer_id)
    # Reuse validated join on observed complete rows; stale clients use popularity.
    observed=np.array([cid in current_ids for cid in ids])
    meta=pd.DataFrame({'customer_id':ids}).merge(metadata_month(RECOMMEND_MONTH),on='customer_id',how='left',validate='one_to_one')
    score=score_batch(final_model,X,meta,SELECTED_METHOD)
    stale=np.array(status)!='current_complete'
    score[stale]=score_batch(final_model,X[stale],meta.loc[stale].reset_index(drop=True),'popularity')
    ranks,values=rank_scores(score)
    write_recs(RECOMMEND_MONTH,SELECTED_METHOD,ids,ranks,values,start)
    pd.DataFrame({'customer_id':ids,'catalog_status':status,
                  'scoring_method':np.where(stale,'popularity_fallback',SELECTED_METHOD)}).to_parquet(
        OUT/'product_recommendation'/RECOMMEND_MONTH/SELECTED_METHOD/f'status_{start:07d}.parquet',index=False)
print('Đã xuất gợi ý cho',len(universe),'khách; catalog_status lưu riêng để kiểm tra')


## Tóm tắt đầu ra và giới hạn
- Cell 6: quy mô, lifecycle và độ bao phủ RFM ở tháng gợi ý.
- Cell 13: affinity matrices, temporal rates và group-product rates.
- Cell 17: validation, phương pháp khoá.
- Cell 19: test MAP@7 và phân tích HasEvent/NoObservedEvent/lifecycle/bucket RFM.
- Cell 22: recommendation mọi khách, kèm lifecycle, RFM nullable và catalog_status.

Các input phải là output Parquet/JSON từ phiên notebook đã chạy; file .ipynb không chứa các partition này. RFM dùng R + F_event_months + M_group_points, lifecycle độc lập với RFM. Các trọng số là baseline cố định; không mặc định ghép cả ba sẽ tăng chất lượng. Không coi missing snapshot là label 0; raw portfolio snapshot cần đồng bộ với feature exports.
